# Authenfy: EfficientNet-B0 + DCT + pHash into an MLP

Real training and controlled all-frame comparison on a Colab T4. Uses Kaggle `xdxd003/ff-c23` v1, official source-pair splits, 4,320 training / 840 validation / 240 fixed held-out videos. This is not the complete Kaggle test collection. Test scores never select a model or threshold. Fully AI-generated video detection is a separate v2 task.


In [ ]:
import torch, torchvision, platform
print({'device':torch.cuda.get_device_name(0),'torch':torch.__version__,'torchvision':torchvision.__version__,'python':platform.python_version()})
assert 'T4' in torch.cuda.get_device_name(0)


In [ ]:
import os, subprocess, sys
from pathlib import Path
if not Path('/content/authenfy/.git').exists():
    subprocess.check_call(['git','clone','https://github.com/vibhanshu-s/authenfy.git','/content/authenfy'])
os.chdir('/content/authenfy')
subprocess.check_call(['git','fetch','origin','feature/efficientnet-fusion-v1'])
subprocess.check_call(['git','checkout','ac81f7c5f43de9c53da815dc1974542b5ea75f37'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','scikit-learn==1.7.2','onnx==1.19.1','onnxruntime==1.31.0','psutil==7.2.2','matplotlib==3.10.7'])
os.environ['OMP_NUM_THREADS']='2'
os.environ['MKL_NUM_THREADS']='2'
def run(command):
    with Path('/content/authenfy-pipeline.log').open('a') as log:
        p=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            print(line,end='',flush=True)
            log.write(line); log.flush()
        if p.wait()!=0:
            raise RuntimeError('Stage failed; see /content/authenfy-pipeline.log')


## Data

One bulk download avoids Kaggle per-file request throttling. Allow about 40 GiB for archive and selected videos, plus crop cache. Google DeepFakeDetection is excluded because its actor identifiers require a separate split mapping. Dataset terms: https://github.com/ondyari/FaceForensics


In [ ]:
run([sys.executable,'scripts/prepare_ff_c23.py','data/ff-c23','--test-pairs','20'])


## Train, then evaluate

Eight baseline epochs and twenty epochs per MLP; validation video log loss selects checkpoints. The backbone is frozen before fitting deep-only and fused heads. Timing uses complete clips, three repeats, rotated arm order, synchronized FP32 inference and separately recorded model loading.


In [ ]:
run([sys.executable,'-m','training.train','data/ff-c23/manifest.csv','runs/ff-c23','--require-t4'])


In [ ]:
run([sys.executable,'-m','training.benchmark','data/ff-c23/manifest.csv','runs/ff-c23','--require-t4'])


In [ ]:
run([sys.executable,'-m','training.export','data/ff-c23/manifest.csv','runs/ff-c23'])
run([sys.executable,'-m','training.plots','runs/ff-c23'])


## Preserve all results

Bundle includes checkpoint weights, ONNX/card, complete metrics, CSV predictions/timings, compressed frame scores, plots, manifest, provenance and logs. It excludes source media and crop cache. Accuracy is specific to these held-out source pairs; no identity-disjoint or production claim.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
bundle=Path('/content/authenfy-efficientnet-v1.zip')
with ZipFile(bundle,'w',compression=ZIP_DEFLATED) as z:
    for p in Path('runs/ff-c23').rglob('*'):
        if p.is_file() and 'crop-cache' not in p.parts:
            z.write(p, str(p.relative_to('runs/ff-c23')))
    for p in [Path('data/ff-c23/manifest.csv'),Path('data/ff-c23/provenance.json'),Path('/content/authenfy-pipeline.log')]:
        z.write(p,p.name)
print(bundle, bundle.stat().st_size)


In [ ]:
from google.colab import files
files.download(str(bundle))
